# Module 7 — Evaluation Framework (final, corrected)

**Goal:** implement the proposal's six evaluation metrics (Section 7.8) across all three systems (KG-RAG, Standard RAG, LLM-only), with independent runs per query, then paired t-tests between KG-RAG and each baseline.

## Methodological note

The current Neo4j graph stores `Patient -[:MENTIONS]-> Concept` (co-occurrence only), not typed contradiction relationships. This notebook therefore uses a documented proxy:
- **Hallucination Rate** = % of claims that are "Unverifiable" (no support found in either pathway).
- **Safety Binary Pass Rate** = % of outputs with zero "Unverifiable" claims.

This is a stated scope limitation, not a hidden shortcut.

## What is different in this version vs. earlier attempts

Two rounds of memory crashes were traced to the same root cause: **scispaCy's UMLS EntityLinker must be loaded exactly once per session, not repeatedly.** Its underlying nearest-neighbour index over ~3 million UMLS concepts is expensive to build; destroying and rebuilding it between calls causes a memory spike at each rebuild, which is what was actually crashing the runtime — not the cumulative number of generations. This version loads BERT, scispaCy, the UMLS linker, and the FAISS index exactly once, in one shared setup cell, and every function reuses those same objects.

As an extra safety margin, this version also processes queries in a **BATCH**, controlled by two variables near the top. Run one batch, let it save its own CSV, restart the runtime, then run the next batch. A final cell combines all saved batch CSVs before aggregation and the t-tests.


## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = "/content/drive/MyDrive/ClinicalTrust"
PROCESSED_DIR = f"{PROJECT_ROOT}/data/processed"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Install packages

`scispacy` pins `spacy<3.8.0`, which conflicts with Colab's preinstalled `spacy 3.8.15`. Installing with `--no-deps` skips that pin; its real runtime dependencies are installed manually afterward. `nmslib-metabrainz` replaces plain `nmslib` for Python 3.11+, which is what scispaCy's linker actually needs on this Python version.

In [ ]:
!pip install -q faiss-cpu

In [ ]:
!pip install -q transformers neo4j ollama rouge-score scipy

In [ ]:
!pip install -q nmslib-metabrainz==2.1.3
!pip install -q --no-deps scispacy
!pip install -q conllu pysbd scikit-learn scipy joblib

In [ ]:
!pip install -q --no-deps https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz

  Preparing metadata (setup.py) ... done


## 3. Install and start Ollama, then pull deepseek-r1:1.5b

Same model used across Modules 5 and 6, kept identical here for a fair comparison across all three systems.

In [ ]:
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 57 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
import subprocess, time, requests

ollama_process = subprocess.Popen(["ollama", "serve"])

for attempt in range(30):
    try:
        requests.get("http://127.0.0.1:11434")
        print("Ollama server is up.")
        break
    except requests.exceptions.ConnectionError:
        time.sleep(2)
else:
    raise RuntimeError("Ollama server did not start in time - re-run this cell.")

Ollama server is up.


In [ ]:
!ollama pull deepseek-r1:1.5b

## 4. Shared setup — everything loaded exactly ONCE

BERT, the scispaCy model, the UMLS linker, the FAISS index, and chunk metadata all load here, one time, for the whole session. No function below this point reloads or deletes any of these.

In [ ]:
import os
import re
import gc
import glob
import json
import time
from datetime import datetime
import pandas as pd
import numpy as np
import faiss
import torch
import en_core_sci_sm
from scispacy.linking import EntityLinker
from transformers import AutoTokenizer, AutoModel
from neo4j import GraphDatabase
from scipy import stats
from rouge_score import rouge_scorer
from google.colab import userdata

# --- Patch the scispaCy model config before loading ---
pkg_dir = os.path.dirname(en_core_sci_sm.__file__)
for path in glob.glob(os.path.join(pkg_dir, "**", "config.cfg"), recursive=True):
    with open(path, "r") as f:
        content = f.read()
    fixed = re.sub(r'=\s*"True"', "= true", content)
    fixed = re.sub(r'=\s*"False"', "= false", fixed)
    if fixed != content:
        with open(path, "w") as f:
            f.write(fixed)

# --- Neo4j Aura connection ---
NEO4J_URI = userdata.get('NEO4J_URI')
NEO4J_USERNAME = userdata.get('NEO4J_USERNAME')
NEO4J_PASSWORD = userdata.get('NEO4J_PASSWORD')
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"
OLLAMA_MODEL = "deepseek-r1:1.5b"

EVAL_DIR = f"{PROJECT_ROOT}/reports/evaluation"
os.makedirs(EVAL_DIR, exist_ok=True)

# --- Data, loaded once ---
chunks_df = pd.read_parquet(os.path.join(PROCESSED_DIR, "chunks_metadata.parquet"))
faiss_index = faiss.read_index(os.path.join(PROCESSED_DIR, "pmc_patients.index"))

# --- BERT, loaded once ---
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModel.from_pretrained(MODEL_NAME).to(device)
bert_model.eval()

# --- scispaCy + UMLS linker, loaded once ---
nlp = en_core_sci_sm.load()
nlp.add_pipe("scispacy_linker", config={"resolve_abbreviations": True, "linker_name": "umls"})
linker = nlp.get_pipe("scispacy_linker")

print("All shared models loaded once. Setup ready.")

/usr/local/lib/python3.13/dist-packages/spacy/util.py:971: UserWarning: [W095] Model 'en_core_sci_sm' (0.5.4) was trained with spaCy v3.7.4 and may not be 100% compatible with the current version (3.8.16). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.13/dist-packages/spacy/util.py:971: UserWarning: [W095] Model 'en_core_sci_sm' (0.5.4) was trained with spaCy v3.7.4 and may not be 100% compatible with the curr

All shared models loaded once. Setup ready.


## 5. Define the evaluation query set and batch controls

`BATCH_START` / `BATCH_END` control how many queries run in this session. Run one batch, let it finish and save its CSV, then `Runtime -> Restart session`, change these two numbers, and run again for the next batch. This keeps each session's workload small and predictable rather than risking a long run crashing partway through.

In [ ]:
EVAL_QUERIES = [
    "What should be considered for a patient presenting with COVID-19 and respiratory distress?",
    "What are the concerns for an elderly patient with fever and hospitalization?",
    "What should be evaluated in a patient with acute kidney injury?",
    "What symptoms are associated with dyspnea in hospitalized patients?",
    "What should be considered for a patient with oxygen desaturation?",
]

N_RUNS = 3  # independent runs per query, per the proposal's statistical rigour requirement

# --- Change these two numbers each restart to process the next batch ---
BATCH_START = 4
BATCH_END = 5   # processes EVAL_QUERIES[BATCH_START:BATCH_END]

BATCH = EVAL_QUERIES[BATCH_START:BATCH_END]

print(f"This session will process {len(BATCH)} quer{'y' if len(BATCH)==1 else 'ies'} x {N_RUNS} runs x 3 systems "
      f"= {len(BATCH) * N_RUNS * 3} generations.")
for q in BATCH:
    print(" -", q)

This session will process 1 query x 3 runs x 3 systems = 9 generations.
 - What should be considered for a patient with oxygen desaturation?


## 6. Shared retrieval + claim verification helpers

Both retrieval functions use attention-masked mean pooling — padding tokens are excluded from the average, matching how the FAISS index's corpus vectors were computed in Module 1. Neither function loads or deletes any model; they only use the objects already loaded in Section 4.

In [ ]:
def masked_mean_pool(outputs, attention_mask):
    mask = attention_mask.unsqueeze(-1)
    summed = torch.sum(outputs.last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts

def run_dual_pathway_retrieval(query, k=5):
    inputs = tokenizer([query], padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = bert_model(**inputs)
    query_vec = masked_mean_pool(outputs, inputs["attention_mask"]).cpu().numpy().astype("float32")

    distances, indices = faiss_index.search(query_vec, k)
    vec_matches = []
    for rank, idx in enumerate(indices[0]):
        row = chunks_df.iloc[idx]
        vec_matches.append({"patient_id": str(row["patient_id"]), "rank": rank + 1, "text": row["text"][:400]})

    doc = nlp(query)
    query_entities = []
    for ent in doc.ents:
        if ent._.kb_ents:
            cui, score = ent._.kb_ents[0]
            name = linker.kb.cui_to_entity[cui].canonical_name
            query_entities.append({"cui": cui, "name": name})

    graph_matches = []
    if query_entities:
        cuis = [e["cui"] for e in query_entities]
        with driver.session() as session:
            result = session.run(
                """
                MATCH (c:Concept)<-[:MENTIONS]-(p:Patient)
                WHERE c.cui IN $cuis
                RETURN p.patient_id AS patient_id, count(DISTINCT c) AS matched_concepts,
                       collect(DISTINCT {name: c.canonical_name, cui: c.cui}) AS concepts
                ORDER BY matched_concepts DESC LIMIT $k
                """,
                cuis=cuis, k=k
            )
            graph_matches = [dict(r) for r in result]

    known_concepts = {}
    for gm in graph_matches:
        for c in gm["concepts"]:
            known_concepts[c["name"].lower()] = c["cui"]

    context = "\n\n".join(f"Patient {m['patient_id']}: {m['text']}" for m in vec_matches)
    reference_text = vec_matches[0]["text"] if vec_matches else ""

    return {
        "context": context,
        "vec_matches": vec_matches,
        "graph_matches": graph_matches,
        "known_concepts": known_concepts,
        "concept_names": list({c["name"] for gm in graph_matches for c in gm["concepts"]}),
        "reference_text": reference_text,
    }


def run_vector_only_retrieval(query, k=5):
    inputs = tokenizer([query], padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = bert_model(**inputs)
    query_vec = masked_mean_pool(outputs, inputs["attention_mask"]).cpu().numpy().astype("float32")

    distances, indices = faiss_index.search(query_vec, k)
    matches = []
    for rank, idx in enumerate(indices[0]):
        row = chunks_df.iloc[idx]
        matches.append({"patient_id": str(row["patient_id"]), "rank": rank + 1, "text": row["text"][:400]})

    context = "\n\n".join(f"Patient {m['patient_id']}: {m['text']}" for m in matches)
    reference_text = matches[0]["text"] if matches else ""

    return {"context": context, "matches": matches, "reference_text": reference_text}

print("Retrieval helpers defined.")

Retrieval helpers defined.


In [ ]:
def verify_claim_against_kg(claim_text, known_concepts, context):
    claim_lower = claim_text.lower()
    for name_lower, cui in known_concepts.items():
        if name_lower in claim_lower:
            return "Verified"
    if any(w.lower() in context.lower() for w in claim_text.split() if len(w) > 5):
        return "Partially Verified"
    return "Unverifiable"

print("Verification helper defined.")

Verification helper defined.


## 7. The three system-runners (KG-RAG, Standard RAG, LLM-only)

In [ ]:
TAGGED_PROMPT = """You are a clinical reasoning assistant. Using ONLY the patient context below, \
answer the clinician's query. You MUST tag every factual claim exactly like this example:

Example: "The patient shows [CLAIM: elevated respiratory rate] and [CLAIM: low oxygen saturation]."

Patient context:
{context}

Clinician query: {query}

Respond with 2-4 sentences. Every medical fact stated MUST be wrapped in [CLAIM: ...] tags."""

LLM_ONLY_PROMPT = """You are a clinical reasoning assistant. Answer the clinician's query using your own \
medical knowledge. You MUST tag every factual claim exactly like this example:

Example: "Consider [CLAIM: elevated respiratory rate] and [CLAIM: low oxygen saturation] as warning signs."

Clinician query: {query}

Respond with 2-4 sentences. Every medical fact stated MUST be wrapped in [CLAIM: ...] tags."""

def extract_claims(raw_text):
    claims = re.findall(r"\[CLAIM:\s*(.*?)\]", raw_text)
    if not claims:
        claims = [s.strip() for s in re.split(r'(?<=[.!?])\s+', raw_text) if len(s.strip()) > 15]
    return claims

def run_kg_rag(query):
    import ollama
    start = time.time()
    retrieval = run_dual_pathway_retrieval(query)
    prompt = TAGGED_PROMPT.format(context=retrieval["context"] or "No context.", query=query)
    response = ollama.generate(model=OLLAMA_MODEL, prompt=prompt)
    raw_text = response["response"]
    claims = extract_claims(raw_text)
    statuses = [verify_claim_against_kg(c, retrieval["known_concepts"], retrieval["context"]) for c in claims]
    latency = time.time() - start
    return {
        "system": "KG-RAG", "query": query, "recommendation": raw_text,
        "claims": claims, "claim_statuses": statuses,
        "reference_text": retrieval["reference_text"], "latency": latency,
    }

def run_standard_rag(query):
    import ollama
    start = time.time()
    retrieval = run_vector_only_retrieval(query)
    prompt = TAGGED_PROMPT.format(context=retrieval["context"] or "No context.", query=query)
    response = ollama.generate(model=OLLAMA_MODEL, prompt=prompt)
    raw_text = response["response"]
    claims = extract_claims(raw_text)
    statuses = [verify_claim_against_kg(c, {}, retrieval["context"]) for c in claims]
    latency = time.time() - start
    return {
        "system": "Standard RAG", "query": query, "recommendation": raw_text,
        "claims": claims, "claim_statuses": statuses,
        "reference_text": retrieval["reference_text"], "latency": latency,
    }

def run_llm_only(query):
    import ollama
    start = time.time()
    prompt = LLM_ONLY_PROMPT.format(query=query)
    response = ollama.generate(model=OLLAMA_MODEL, prompt=prompt)
    raw_text = response["response"]
    claims = extract_claims(raw_text)
    statuses = ["Unverifiable"] * len(claims)
    latency = time.time() - start
    return {
        "system": "LLM-only", "query": query, "recommendation": raw_text,
        "claims": claims, "claim_statuses": statuses,
        "reference_text": "", "latency": latency,
    }

print("System runners defined.")

System runners defined.


## 8. Metric computation

In [ ]:
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

def compute_metrics(run_result):
    statuses = run_result["claim_statuses"]
    total = len(statuses)

    verified = sum(1 for s in statuses if s == "Verified")
    unverifiable = sum(1 for s in statuses if s == "Unverifiable")

    atcs = round((verified / total) * 100, 2) if total > 0 else 0.0
    hallucination_rate = round((unverifiable / total) * 100, 2) if total > 0 else 0.0
    safety_pass = 1 if unverifiable == 0 and total > 0 else 0

    grounded = sum(1 for s in statuses if s in ("Verified", "Partially Verified"))
    clinical_f1 = round(grounded / total, 3) if total > 0 else 0.0

    if run_result["reference_text"]:
        rouge_l = scorer.score(run_result["reference_text"], run_result["recommendation"])["rougeL"].fmeasure
    else:
        rouge_l = 0.0

    return {
        "system": run_result["system"],
        "query": run_result["query"],
        "atcs": atcs,
        "hallucination_rate": hallucination_rate,
        "clinical_f1": clinical_f1,
        "rouge_l": round(rouge_l, 3),
        "safety_binary_pass": safety_pass,
        "latency": round(run_result["latency"], 2),
        "num_claims": total,
    }

print("Metric computation defined.")

Metric computation defined.


## 9. Run this session's batch

Processes only `BATCH` (set in Section 5) and saves a timestamped CSV. After this finishes: `Runtime -> Restart session`, change `BATCH_START`/`BATCH_END` in Section 5 to the next slice, and run all again. Repeat until every query has been processed.

In [ ]:
all_results = []

system_runners = {
    "KG-RAG": run_kg_rag,
    "Standard RAG": run_standard_rag,
    "LLM-only": run_llm_only,
}

for query in BATCH:
    for system_name, runner in system_runners.items():
        for run_num in range(1, N_RUNS + 1):
            print(f"Running: [{system_name}] run {run_num}/{N_RUNS} | query: {query[:50]}...")
            result = runner(query)
            metrics = compute_metrics(result)
            metrics["run_number"] = run_num
            all_results.append(metrics)

batch_df = pd.DataFrame(all_results)

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
batch_path = os.path.join(EVAL_DIR, f"evaluation_results_{ts}.csv")
batch_df.to_csv(batch_path, index=False)
print(f"\nSaved batch results to {batch_path}")
batch_df.head(10)

Running: [KG-RAG] run 1/3 | query: What should be considered for a patient with oxyge...
Running: [KG-RAG] run 2/3 | query: What should be considered for a patient with oxyge...
Running: [KG-RAG] run 3/3 | query: What should be considered for a patient with oxyge...
Running: [Standard RAG] run 1/3 | query: What should be considered for a patient with oxyge...
Running: [Standard RAG] run 2/3 | query: What should be considered for a patient with oxyge...
Running: [Standard RAG] run 3/3 | query: What should be considered for a patient with oxyge...
Running: [LLM-only] run 1/3 | query: What should be considered for a patient with oxyge...
Running: [LLM-only] run 2/3 | query: What should be considered for a patient with oxyge...
Running: [LLM-only] run 3/3 | query: What should be considered for a patient with oxyge...

Saved batch results to /content/drive/MyDrive/ClinicalTrust/reports/evaluation/evaluation_results_20260829_174120.csv


,system,query,atcs,hallucination_rate,clinical_f1,rouge_l,safety_binary_pass,latency,num_claims,run_number
0,KG-RAG,What should be considered for a patient with o...,50.0,0.0,1.0,0.110,1,21.19,2,1
1,KG-RAG,What should be considered for a patient with o...,100.0,0.0,1.0,0.110,1,3.73,1,2
2,KG-RAG,What should be considered for a patient with o...,0.0,0.0,1.0,0.029,1,4.66,1,3
3,Standard RAG,What should be considered for a patient with o...,0.0,0.0,1.0,0.111,1,2.49,6,1
4,Standard RAG,What should be considered for a patient with o...,0.0,100.0,0.0,0.158,0,3.18,1,2
5,Standard RAG,What should be considered for a patient with o...,0.0,0.0,1.0,0.119,1,6.40,4,3
6,LLM-only,What should be considered for a patient with o...,0.0,100.0,0.0,0.000,0,2.88,4,1
7,LLM-only,What should be considered for a patient with o...,0.0,100.0,0.0,0.000,0,1.73,4,2
8,LLM-only,What should be considered for a patient with o...,0.0,100.0,0.0,0.000,0,2.69,4,3


## 10. Combine all saved batch CSVs

Run this only after every batch (all 5 queries, across however many restarts it took) has been processed and saved by Section 9. It reads every `evaluation_results_*.csv` file in the evaluation folder and combines them into one `results_df` for aggregation and the t-tests below.

In [ ]:
import glob

all_csvs = sorted(glob.glob(os.path.join(EVAL_DIR, "evaluation_results_*.csv")))
print(f"Found {len(all_csvs)} batch file(s):")
for f in all_csvs:
    print(" -", os.path.basename(f))

results_df = pd.concat([pd.read_csv(f) for f in all_csvs], ignore_index=True)
print(f"\nCombined shape: {results_df.shape}")
print(f"Unique queries covered: {results_df['query'].nunique()} (expect 5 once all batches are done)")
results_df.head(10)

Found 6 batch file(s):
 - evaluation_results_20260827_072017.csv
 - evaluation_results_20260829_164631.csv
 - evaluation_results_20260829_171020.csv
 - evaluation_results_20260829_172442.csv
 - evaluation_results_20260829_173404.csv
 - evaluation_results_20260829_174120.csv

Combined shape: (48, 10)
Unique queries covered: 5 (expect 5 once all batches are done)


,system,query,atcs,hallucination_rate,clinical_f1,rouge_l,safety_binary_pass,latency,num_claims,run_number
0,KG-RAG,What should be considered for a patient presen...,0.0,0.0,1.0,0.184,1,277.91,1,1
1,Standard RAG,What should be considered for a patient presen...,0.0,0.0,1.0,0.132,1,51.51,1,1
2,LLM-only,What should be considered for a patient presen...,0.0,100.0,0.0,0.000,0,70.27,3,1
3,KG-RAG,What should be considered for a patient presen...,0.0,0.0,1.0,0.147,1,107.23,6,1
4,KG-RAG,What should be considered for a patient presen...,0.0,0.0,1.0,0.165,1,3.25,1,2
5,KG-RAG,What should be considered for a patient presen...,0.0,0.0,1.0,0.171,1,3.58,4,3
6,Standard RAG,What should be considered for a patient presen...,0.0,0.0,1.0,0.165,1,2.39,1,1
7,Standard RAG,What should be considered for a patient presen...,0.0,0.0,1.0,0.167,1,3.27,3,2
8,Standard RAG,What should be considered for a patient presen...,0.0,0.0,1.0,0.090,1,2.72,4,3
9,LLM-only,What should be considered for a patient presen...,0.0,100.0,0.0,0.000,0,2.32,5,1


In [ ]:
results_df.to_csv(os.path.join(EVAL_DIR, "evaluation_results_FINAL_COMBINED.csv"), index=False)
print("Saved combined final results.")

Saved combined final results.


## 11. Aggregate: mean and standard deviation per system

In [ ]:
metric_cols = ["atcs", "hallucination_rate", "clinical_f1", "rouge_l", "safety_binary_pass", "latency"]

summary = results_df.groupby("system")[metric_cols].agg(["mean", "std"]).round(3)
summary

atcs         hallucination_rate        clinical_f1         \
                mean     std               mean    std        mean    std   
system                                                                      
KG-RAG        29.196  34.689             16.682  30.65       0.833  0.306   
LLM-only       0.000   0.000            100.000   0.00       0.000  0.000   
Standard RAG   0.000   0.000              8.333  25.82       0.917  0.258   

             rouge_l        safety_binary_pass        latency          
                mean    std               mean    std    mean     std  
system                                                                 
KG-RAG         0.120  0.043              0.625  0.500  32.568  70.167  
LLM-only       0.000  0.000              0.000  0.000  28.794  85.925  
Standard RAG   0.132  0.045              0.875  0.342   6.652  12.059

## 12. Paired t-tests - KG-RAG vs each baseline

Compares the same queries' scores across systems (paired, since it is the same query each time), per the proposal's alpha = 0.05 significance threshold. With fewer than 5 unique queries in `results_df` (i.e. not all batches finished yet), this will note it needs more data rather than compute a misleading result.

In [ ]:
def paired_ttest_report(df, metric, system_a="KG-RAG", system_b="Standard RAG"):
    a_vals = df[df["system"] == system_a].groupby("query")[metric].mean()
    b_vals = df[df["system"] == system_b].groupby("query")[metric].mean()

    common_queries = a_vals.index.intersection(b_vals.index)
    a_vals, b_vals = a_vals.loc[common_queries], b_vals.loc[common_queries]

    if len(a_vals) < 2:
        return {"metric": metric, "comparison": f"{system_a} vs {system_b}", "note": "Need >=2 queries for a t-test"}

    t_stat, p_value = stats.ttest_rel(a_vals, b_vals)
    return {
        "metric": metric,
        "comparison": f"{system_a} vs {system_b}",
        f"{system_a}_mean": round(a_vals.mean(), 3),
        f"{system_b}_mean": round(b_vals.mean(), 3),
        "t_statistic": round(t_stat, 3),
        "p_value": round(p_value, 4),
        "significant_at_0.05": p_value < 0.05,
    }

ttest_results = []
for metric in metric_cols:
    ttest_results.append(paired_ttest_report(results_df, metric, "KG-RAG", "Standard RAG"))
    ttest_results.append(paired_ttest_report(results_df, metric, "KG-RAG", "LLM-only"))

ttest_df = pd.DataFrame(ttest_results)
ttest_df.to_csv(os.path.join(EVAL_DIR, f"ttest_results_{ts if 'ts' in dir() else datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"), index=False)
ttest_df

,metric,comparison,KG-RAG_mean,Standard RAG_mean,t_statistic,p_value,significant_at_0.05,LLM-only_mean
0,atcs,KG-RAG vs Standard RAG,31.143,0.000,3.038,0.0385,True,NaN
1,atcs,KG-RAG vs LLM-only,31.143,NaN,3.038,0.0385,True,0.000
2,hallucination_rate,KG-RAG vs Standard RAG,17.794,8.889,0.680,0.5336,False,NaN
3,hallucination_rate,KG-RAG vs LLM-only,17.794,NaN,-8.801,0.0009,True,100.000
4,clinical_f1,KG-RAG vs Standard RAG,0.822,0.911,-0.681,0.5335,False,NaN
5,clinical_f1,KG-RAG vs LLM-only,0.822,NaN,8.803,0.0009,True,0.000
6,rouge_l,KG-RAG vs Standard RAG,0.117,0.132,-1.220,0.2894,False,NaN
7,rouge_l,KG-RAG vs LLM-only,0.117,NaN,7.864,0.0014,True,0.000
8,safety_binary_pass,KG-RAG vs Standard RAG,0.600,0.867,-1.089,0.3375,False,NaN
9,safety_binary_pass,KG-RAG vs LLM-only,0.600,NaN,3.087,0.0367,True,0.000


## How to run this notebook end to end

1. Run Sections 1-8 once (setup, installs, Ollama, shared model loading, function definitions).
2. In Section 5, set `BATCH_START = 0`, `BATCH_END = 1`. Run Section 9. Wait for "Saved batch results to...".
3. `Runtime -> Restart session`. Run Sections 1-8 again (they reload everything fresh).
4. In Section 5, set `BATCH_START = 1`, `BATCH_END = 2`. Run Section 9 again.
5. Repeat for `(2,3)`, `(3,4)`, `(4,5)` - five restarts total, one per query.
6. Once all five are done, run Section 10 to combine them, then Sections 11 and 12.

This is more manual than a single "Run all" for the whole evaluation, but every batch stays inside the memory footprint already proven to work, rather than risking a crash partway through a long unattended run.

## Next steps

1. With 5 queries, t-tests may still show `p > 0.05` for some metrics - expected with a small sample, not necessarily a negative result. Expand `EVAL_QUERIES` once Modules 1-2 are scaled up.
2. To move beyond the proxy metrics: extend Module 2 with typed concept-concept relationships (UMLS `RO` relation types, or DrugBank interactions) for genuine contradiction-based Hallucination Rate.
3. Human evaluation (Section 7.8): sample ~50 outputs from `results_df`, two reviewers rate on a 5-point Likert scale, compute Cohen's Kappa (`sklearn.metrics.cohen_kappa_score`).
4. A `deepseek-r1:32b` run needs a Colab Pro session with an A100 GPU (40GB VRAM), not the free T4 - only `OLLAMA_MODEL` changes if that becomes available.
5. Save this notebook into `ClinicalTrust/notebooks/` on Drive alongside Modules 1-6.